# UCSF Manifest

The notebook `01_metadata_manifest.ipynb` already does a lot of the metadata cleanup for UCSF, which is saved as `Data/ucsf_clean.csv`. However, `process_chunk` takes the full patient folder name and needs the patient id in the manifest to match. In the cleaned metadata file, the patient id lacks the "_nifti" suffix of the patient folder. This notebook merely adds this suffix and saves the new file as `manifest_ucsf.csv`. 

In [ ]:
import pandas as pd
from src.config import DATA, RAW_UCSF, MANIFEST_UCSF


clean = pd.read_csv(DATA/ "ucsf_clean.csv")
print(clean.shape)
print(list(clean.columns))
print(clean.iloc[:, 0].head(8).tolist())

(495, 4)
['patient_id', 'age', 'sex', 'idh']
['UCSF-PDGM-0004', 'UCSF-PDGM-0005', 'UCSF-PDGM-0007', 'UCSF-PDGM-0008', 'UCSF-PDGM-0009', 'UCSF-PDGM-0010', 'UCSF-PDGM-0011', 'UCSF-PDGM-0012']


In [ ]:
# Sanity checks on the IDs and columns (no labels grouped or printed)
print("unique IDs:", clean["patient_id"].is_unique)                                  # True
print("all 4-digit format:", clean["patient_id"].str.fullmatch(r"UCSF-PDGM-\d{4}").all())  # True
print("any follow-up IDs:", clean["patient_id"].str.contains("FU").any())            # False
print("age range:", clean["age"].min(), clean["age"].max())   # brief says 17 to 94 (from the dataset page)
print(clean["sex"].value_counts(dropna=False))                # counts only: check the coding before mapping later
print(clean[["age", "sex", "idh"]].isna().sum())              # expect 0 in each column

unique IDs: True
all 4-digit format: True
any follow-up IDs: False
age range: 17 94
sex
M    296
F    199
Name: count, dtype: int64
age    0
sex    0
idh    0
dtype: int64


In [13]:
manifest = clean.rename(columns={"patient_id": "case_id"})
manifest.insert(0, "patient_id", manifest["case_id"] + "_nifti")   # exact folder name
manifest.to_csv(MANIFEST_UCSF, index=False)

on_disk = {p.name for p in RAW_UCSF.iterdir() if p.is_dir()}
print(f"{len(manifest)} rows | downloaded folders all in manifest: {on_disk <= set(manifest['patient_id'])}")

495 rows | downloaded folders all in manifest: True
